# ML102 — Drug Adoption Prediction

## Final End-to-End Pipeline

**Final model uses exactly 18 SHAP-selected features.**

- Rolling window CV = 3
- Model selection = CV PR-AUC
- Threshold selection = OOF F1.5
- No calibration
- No fixed 0.5 threshold
- No business-defined threshold
- Final evaluation on historical out-of-time test data
- Final ROC-AUC and PR-AUC curves


## Imports and Configuration

In [ ]:
# ML102 FINAL END-TO-END PIPELINE
# Final model: exactly 18 SHAP-selected features
# Rolling CV = 3
# Model selection metric = PR-AUC
# Threshold = OOF F1.5 (recall weighted more than precision)
# No calibration / no fixed 0.5 threshold / no business threshold

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb
import lightgbm as lgb
import shap

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, roc_curve, auc,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
N_FINAL_FEATURES = 18
CV_N_SPLITS = 3
FBETA_BETA = 1.5

FILE1_PATH = "Input_data_file1.csv"
FILE2_PATH = "Input_data_file2.csv"
TEST_PHYS_PATH = "Test_physicians.csv"
OUTPUT_PATH = "Test_physicians_scored_Q11.csv"

ID_COL = "physician_id"
TIME_COL = "year_quarter"
TARGET_COL = "brand_prescribed"

TRAIN_QUARTER_START = 201504
TRAIN_QUARTER_END = 201701
TEST_QUARTER_START = 201702
TEST_QUARTER_END = 201703
SCORE_QUARTER = 201704

# Raw variables used to create business-sense lag/average/change features.
LAG_AVG_CHANGE_COLS = [
    "total_representative_visits",
    "total_sample_dropped",
    "total_prescriptions_for_indication1",
    "total_prescriptions_for_indication2",
    "total_prescriptions_for_indication3",
    "total_patient_with_commercial_insurance_plan",
    "total_patient_with_medicare_insurance_plan",
    "total_patient_with_medicaid_insurance_plan",
    "brand_web_impressions",
    "brand_ehr_impressions",
    "brand_enews_impressions",
    "brand_mobile_impressions",
    "total_competitor_prescription",
    "new_prescriptions",
]

LAG_CHANGE_ONLY_COLS = [
    "saving_cards_dropped",
    "vouchers_dropped",
    "total_seminar_as_attendee",
    "total_seminar_as_speaker",
]

MOST_RECENT_ONLY_COLS = [
    "physician_hospital_affiliation",
    "physician_in_group_practice",
]

TIER_COL = "physician_value_tier"
CATEGORICAL_COLS = ["physician_gender", "physician_speciality"]

NON_FEATURE_COLS = [ID_COL, "target_quarter", TARGET_COL]


def quarter_range(start_code, end_code):
    quarters = []
    year, quarter = divmod(start_code, 100)
    end_year, end_quarter = divmod(end_code, 100)

    while (year, quarter) <= (end_year, end_quarter):
        quarters.append(year * 100 + quarter)
        quarter += 1
        if quarter == 5:
            quarter = 1
            year += 1
    return quarters


TRAIN_QUARTERS = quarter_range(
    TRAIN_QUARTER_START, TRAIN_QUARTER_END
)
TEST_QUARTERS = quarter_range(
    TEST_QUARTER_START, TEST_QUARTER_END
)


def load_data():
    file1 = pd.read_csv(FILE1_PATH)
    file2 = pd.read_csv(FILE2_PATH)
    test_phys = pd.read_csv(TEST_PHYS_PATH)

    file1 = file1.rename(columns={
        "total_reps_visits": "total_representative_visits",
        "saving_card_dropped": "saving_cards_dropped",
    })

    file1 = file1.loc[
        :, ~file1.columns.str.contains("^Unnamed")
    ]

    file1 = file1.sort_values(
        [ID_COL, TIME_COL]
    ).reset_index(drop=True)

    return file1, file2, test_phys


def encode_tier(df):
    df = df.copy()

    if TIER_COL not in df.columns:
        return df, {}

    tiers = df[TIER_COL].dropna().unique().tolist()

    def key(x):
        try:
            return int(str(x).split("-")[0])
        except Exception:
            return str(x)

    tiers = sorted(tiers, key=key)
    mapping = {v: i + 1 for i, v in enumerate(tiers)}
    df[TIER_COL] = df[TIER_COL].map(mapping)

    return df, mapping


def build_lag_features_core(df):
    df = df.sort_values([ID_COL, TIME_COL]).copy()
    grp = df.groupby(ID_COL, group_keys=False)

    out = pd.DataFrame({
        ID_COL: df[ID_COL].values,
        TIME_COL: df[TIME_COL].values,
        TARGET_COL: (
            df[TARGET_COL].values
            if TARGET_COL in df.columns
            else np.nan
        ),
    })

    all_cols = (
        LAG_AVG_CHANGE_COLS
        + LAG_CHANGE_ONLY_COLS
        + MOST_RECENT_ONLY_COLS
        + [TIER_COL]
    )

    for col in all_cols:
        if col not in df.columns:
            continue

        lag1 = grp[col].shift(1)
        lag2 = grp[col].shift(2)

        if col in LAG_AVG_CHANGE_COLS:
            out[f"{col}_lag_avg"] = np.where(
                lag1.notna() & lag2.notna(),
                (lag1 + lag2) / 2,
                np.nan,
            )
            out[f"{col}_lag_change"] = np.where(
                lag1.notna() & lag2.notna(),
                lag1 - lag2,
                np.nan,
            )

        elif col in LAG_CHANGE_ONLY_COLS or col == TIER_COL:
            out[f"{col}_lag_change"] = np.where(
                lag1.notna() & lag2.notna(),
                lag1 - lag2,
                np.nan,
            )

        if col in MOST_RECENT_ONLY_COLS:
            out[col] = lag1

    out["physician_tenure_quarters"] = grp.cumcount().values
    return out


def build_training_panel(file1):
    panel = build_lag_features_core(file1)

    grp = file1.sort_values(
        [ID_COL, TIME_COL]
    ).groupby(ID_COL, group_keys=False)

    adopted_before = (
        grp[TARGET_COL]
        .apply(lambda s: s.shift(1).fillna(0).cumsum() > 0)
        .values
    )

    panel["_adopted_before"] = adopted_before
    panel = panel[~panel["_adopted_before"]].drop(
        columns="_adopted_before"
    )

    panel = panel.rename(
        columns={TIME_COL: "target_quarter"}
    )

    return panel.reset_index(drop=True)


def build_scoring_panel(file1, score_quarter, physician_ids):
    physician_ids = pd.Index(physician_ids).unique()

    real_rows = file1[
        file1[ID_COL].isin(physician_ids)
    ].copy()

    synthetic = pd.DataFrame({ID_COL: physician_ids})
    synthetic[TIME_COL] = score_quarter

    for col in file1.columns:
        if col not in [ID_COL, TIME_COL]:
            synthetic[col] = np.nan

    extended = pd.concat(
        [real_rows, synthetic],
        ignore_index=True
    )

    panel = build_lag_features_core(extended)
    panel = panel.rename(
        columns={TIME_COL: "target_quarter"}
    )

    return panel[
        panel["target_quarter"] == score_quarter
    ].drop(columns=[TARGET_COL], errors="ignore").reset_index(drop=True)


def basic_eda(panel):
    print("\n" + "=" * 70)
    print("BASIC EDA")
    print("=" * 70)

    print("\nOverall shape:", panel.shape)
    print("\nTarget distribution:")
    print(panel[TARGET_COL].value_counts(dropna=False))
    print("\nTarget rate:")
    print(panel[TARGET_COL].mean())

    quarterly = (
        panel.groupby("target_quarter")[TARGET_COL]
        .agg(["count", "sum", "mean"])
        .reset_index()
    )
    quarterly.columns = [
        "quarter", "eligible_physicians",
        "first_time_adopters", "adoption_rate"
    ]

    print("\nFirst-time adopters by quarter:")
    print(quarterly.to_string(index=False))

    plt.figure(figsize=(12, 5))
    plt.plot(
        quarterly["quarter"].astype(str),
        quarterly["first_time_adopters"],
        marker="o"
    )
    plt.title("First-Time Adopters by Quarter")
    plt.xlabel("Quarter")
    plt.ylabel("Number of First-Time Adopters")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(
        "EDA_first_time_adopters_by_quarter.png",
        dpi=150
    )
    plt.close()

    plt.figure(figsize=(12, 5))
    plt.plot(
        quarterly["quarter"].astype(str),
        quarterly["adoption_rate"],
        marker="o"
    )
    plt.title("First-Time Adoption Rate by Quarter")
    plt.xlabel("Quarter")
    plt.ylabel("Adoption Rate")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(
        "EDA_adoption_rate_by_quarter.png",
        dpi=150
    )
    plt.close()

    plt.figure(figsize=(8, 5))
    sns.countplot(
        data=panel,
        x=TARGET_COL
    )
    plt.title("First-Time Adoption Target Distribution")
    plt.tight_layout()
    plt.savefig(
        "EDA_target_distribution.png",
        dpi=150
    )
    plt.close()

    return quarterly


def merge_and_encode(panel, file2, ohe=None, fit=False):
    merged = panel.merge(
        file2,
        on=ID_COL,
        how="left"
    )

    if fit:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
        ohe.fit(merged[CATEGORICAL_COLS])

    encoded = ohe.transform(
        merged[CATEGORICAL_COLS]
    )

    encoded_df = pd.DataFrame(
        encoded,
        columns=ohe.get_feature_names_out(
            CATEGORICAL_COLS
        ),
        index=merged.index
    )

    merged = pd.concat(
        [
            merged.drop(columns=CATEGORICAL_COLS),
            encoded_df
        ],
        axis=1
    )

    return merged, ohe


def split_time(panel):
    train = panel[
        panel["target_quarter"].isin(TRAIN_QUARTERS)
    ].copy()

    test = panel[
        panel["target_quarter"].isin(TEST_QUARTERS)
    ].copy()

    return train, test


def remove_high_missing(train, test, score, threshold=0.20):
    missing = train.isna().mean()

    drop_cols = [
        c for c in missing[missing > threshold].index
        if c not in NON_FEATURE_COLS
    ]

    train = train.drop(columns=drop_cols, errors="ignore")
    test = test.drop(columns=drop_cols, errors="ignore")
    score = score.drop(columns=drop_cols, errors="ignore")

    return train, test, score


def make_feature_matrix(df):
    cols = [
        c for c in df.columns
        if c not in NON_FEATURE_COLS
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    return df[cols].copy(), cols


def rolling_splits(df, n_splits=3):
    quarters = sorted(
        df["target_quarter"].unique()
    )

    validation_quarters = quarters[-n_splits:]
    splits = []

    print("\n" + "=" * 70)
    print("ROLLING CV")
    print("=" * 70)

    for i, q in enumerate(validation_quarters, 1):
        train_idx = df.index[
            df["target_quarter"] < q
        ]
        valid_idx = df.index[
            df["target_quarter"] == q
        ]

        if (
            len(train_idx) > 0
            and len(valid_idx) > 0
            and df.loc[valid_idx, TARGET_COL].nunique() > 1
        ):
            splits.append((train_idx, valid_idx))
            print(
                f"Fold {i}: train before {q} "
                f"-> validate {q}"
            )

    return splits


def get_model(model_type, params, y):
    params = dict(params)

    multiplier = params.pop(
        "spw_multiplier", 1.0
    )

    ratio = (
        (y == 0).sum()
        / max(1, (y == 1).sum())
    )

    params["scale_pos_weight"] = (
        ratio * multiplier
    )

    if model_type == "xgb":
        return xgb.XGBClassifier(
            **params,
            random_state=RANDOM_STATE,
            eval_metric="aucpr"
        )

    return lgb.LGBMClassifier(
        **params,
        random_state=RANDOM_STATE,
        verbosity=-1
    )


def objective_factory(model_type, X, y, splits):
    ratio = (
        (y == 0).sum()
        / max(1, (y == 1).sum())
    )

    def objective(trial):
        multiplier = trial.suggest_float(
            "spw_multiplier", 0.5, 2.0
        )

        scores = []

        for train_idx, valid_idx in splits:
            if model_type == "xgb":
                params = {
                    "n_estimators": trial.suggest_int(
                        "n_estimators", 100, 400
                    ),
                    "max_depth": trial.suggest_int(
                        "max_depth", 3, 7
                    ),
                    "learning_rate": trial.suggest_float(
                        "learning_rate", 0.02, 0.25, log=True
                    ),
                    "subsample": trial.suggest_float(
                        "subsample", 0.7, 1.0
                    ),
                    "colsample_bytree": trial.suggest_float(
                        "colsample_bytree", 0.7, 1.0
                    ),
                    "min_child_weight": trial.suggest_int(
                        "min_child_weight", 1, 8
                    ),
                    "reg_alpha": trial.suggest_float(
                        "reg_alpha", 1e-3, 5, log=True
                    ),
                    "reg_lambda": trial.suggest_float(
                        "reg_lambda", 1e-3, 5, log=True
                    ),
                    "spw_multiplier": multiplier,
                }
            else:
                params = {
                    "n_estimators": trial.suggest_int(
                        "n_estimators", 100, 400
                    ),
                    "num_leaves": trial.suggest_int(
                        "num_leaves", 15, 80
                    ),
                    "learning_rate": trial.suggest_float(
                        "learning_rate", 0.02, 0.25, log=True
                    ),
                    "subsample": trial.suggest_float(
                        "subsample", 0.7, 1.0
                    ),
                    "colsample_bytree": trial.suggest_float(
                        "colsample_bytree", 0.7, 1.0
                    ),
                    "min_child_samples": trial.suggest_int(
                        "min_child_samples", 10, 50
                    ),
                    "reg_alpha": trial.suggest_float(
                        "reg_alpha", 1e-3, 5, log=True
                    ),
                    "reg_lambda": trial.suggest_float(
                        "reg_lambda", 1e-3, 5, log=True
                    ),
                    "spw_multiplier": multiplier,
                }

            model = get_model(
                model_type, params,
                y.loc[train_idx]
            )

            model.fit(
                X.loc[train_idx].fillna(-999),
                y.loc[train_idx]
            )

            prob = model.predict_proba(
                X.loc[valid_idx].fillna(-999)
            )[:, 1]

            p, r, _ = precision_recall_curve(
                y.loc[valid_idx], prob
            )

            scores.append(auc(r, p))

        return float(np.mean(scores))

    return objective


def tune_model(model_type, X, y, splits, n_trials=20):
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE
        )
    )

    study.optimize(
        objective_factory(
            model_type, X, y, splits
        ),
        n_trials=n_trials,
        show_progress_bar=False
    )

    print(
        f"{model_type.upper()} CV PR-AUC: "
        f"{study.best_value:.4f}"
    )

    return study.best_params, study.best_value


def fit_model(model_type, params, X, y):
    model = get_model(
        model_type, params, y
    )

    model.fit(
        X.fillna(-999),
        y
    )

    return model


def evaluate(model, X, y, threshold, title):
    prob = model.predict_proba(
        X.fillna(-999)
    )[:, 1]

    pred = (
        prob >= threshold
    ).astype(int)

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print(
        "Features:",
        X.shape[1]
    )
    print(
        "Threshold:",
        round(threshold, 4)
    )
    print(
        "Accuracy:",
        round(
            accuracy_score(y, pred), 4
        )
    )
    print(
        "Precision:",
        round(
            precision_score(
                y, pred, zero_division=0
            ), 4
        )
    )
    print(
        "Recall:",
        round(
            recall_score(
                y, pred, zero_division=0
            ), 4
        )
    )
    print(
        "F1:",
        round(
            f1_score(
                y, pred, zero_division=0
            ), 4
        )
    )
    print(
        "ROC-AUC:",
        round(
            roc_auc_score(y, prob), 4
        )
    )

    p, r, _ = precision_recall_curve(
        y, prob
    )

    print(
        "PR-AUC:",
        round(
            auc(r, p), 4
        )
    )

    return prob, pred


def oof_predictions(
    model_type, params,
    X, y, splits
):
    oof = pd.Series(
        np.nan,
        index=X.index
    )

    for fold, (
        train_idx,
        valid_idx
    ) in enumerate(splits, 1):

        model = fit_model(
            model_type,
            params,
            X.loc[train_idx],
            y.loc[train_idx]
        )

        oof.loc[valid_idx] = (
            model.predict_proba(
                X.loc[valid_idx].fillna(-999)
            )[:, 1]
        )

        print(
            f"OOF fold {fold} complete"
        )

    return oof.dropna()


def best_fbeta_threshold(
    y, probability, beta=1.5
):
    precision, recall, thresholds = (
        precision_recall_curve(
            y, probability
        )
    )

    b2 = beta ** 2

    score = (
        (1 + b2)
        * precision
        * recall
        / (
            b2 * precision
            + recall
            + 1e-12
        )
    )

    score = score[:-1]

    idx = np.argmax(score)

    return (
        float(thresholds[idx]),
        float(precision[idx]),
        float(recall[idx]),
        float(score[idx])
    )


def shap_top_18(model, X):
    explainer = shap.TreeExplainer(
        model
    )

    shap_values = explainer.shap_values(
        X.fillna(-999)
    )

    if isinstance(shap_values, list):
        shap_values = shap_values[-1]

    mean_abs_shap = np.abs(
        shap_values
    ).mean(axis=0)

    shap_df = pd.DataFrame({
        "feature": X.columns,
        "mean_abs_shap": mean_abs_shap
    }).sort_values(
        "mean_abs_shap",
        ascending=False
    )

    top_features = (
        shap_df.head(
            N_FINAL_FEATURES
        )["feature"].tolist()
    )

    return top_features, shap_df


def plot_final_curves(
    y_test,
    test_probability
):
    fpr, tpr, _ = roc_curve(
        y_test,
        test_probability
    )

    roc_score = roc_auc_score(
        y_test,
        test_probability
    )

    plt.figure(figsize=(7, 6))
    plt.plot(
        fpr, tpr,
        label=f"ROC-AUC = {roc_score:.4f}"
    )
    plt.plot(
        [0, 1], [0, 1],
        linestyle="--"
    )
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Final Model ROC-AUC")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        "FINAL_ROC_AUC.png",
        dpi=150
    )
    plt.close()

    precision, recall, _ = (
        precision_recall_curve(
            y_test,
            test_probability
        )
    )

    pr_score = auc(
        recall, precision
    )

    plt.figure(figsize=(7, 6))
    plt.plot(
        recall,
        precision,
        label=f"PR-AUC = {pr_score:.4f}"
    )
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Final Model Precision-Recall Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        "FINAL_PR_AUC.png",
        dpi=150
    )
    plt.close()

    return roc_score, pr_score


def main():

    # -------------------------------------------------------------------------
    # LOAD
    # -------------------------------------------------------------------------
    file1, file2, test_phys = load_data()
    file1, tier_map = encode_tier(file1)

    # -------------------------------------------------------------------------
    # FEATURE ENGINEERING
    # -------------------------------------------------------------------------
    panel = build_training_panel(file1)

    score_panel = build_scoring_panel(
        file1,
        SCORE_QUARTER,
        test_phys[ID_COL]
    )

    # -------------------------------------------------------------------------
    # BASIC EDA
    # -------------------------------------------------------------------------
    basic_eda(panel)

    # -------------------------------------------------------------------------
    # TIME SPLIT
    # -------------------------------------------------------------------------
    train_panel, test_panel = split_time(
        panel
    )

    train_merged, ohe = merge_and_encode(
        train_panel,
        file2,
        fit=True
    )

    test_merged, _ = merge_and_encode(
        test_panel,
        file2,
        ohe=ohe,
        fit=False
    )

    score_merged, _ = merge_and_encode(
        score_panel,
        file2,
        ohe=ohe,
        fit=False
    )

    # -------------------------------------------------------------------------
    # MISSINGNESS
    # -------------------------------------------------------------------------
    (
        train_merged,
        test_merged,
        score_merged
    ) = remove_high_missing(
        train_merged,
        test_merged,
        score_merged,
        threshold=0.20
    )

    # -------------------------------------------------------------------------
    # FEATURE MATRICES
    # -------------------------------------------------------------------------
    X_train, feature_cols = (
        make_feature_matrix(
            train_merged
        )
    )

    X_test = test_merged[
        feature_cols
    ].copy()

    for c in feature_cols:
        if c not in score_merged.columns:
            score_merged[c] = np.nan

    X_score = score_merged[
        feature_cols
    ].copy()

    y_train = train_merged[
        TARGET_COL
    ].astype(int)

    y_test = test_merged[
        TARGET_COL
    ].astype(int)

    print(
        "\nCandidate features before SHAP:",
        len(feature_cols)
    )

    # -------------------------------------------------------------------------
    # ROLLING CV
    # -------------------------------------------------------------------------
    splits = rolling_splits(
        train_merged,
        CV_N_SPLITS
    )

    # -------------------------------------------------------------------------
    # MODEL TUNING
    # -------------------------------------------------------------------------
    xgb_params, xgb_cv = tune_model(
        "xgb",
        X_train,
        y_train,
        splits,
        n_trials=20
    )

    lgb_params, lgb_cv = tune_model(
        "lgb",
        X_train,
        y_train,
        splits,
        n_trials=20
    )

    if xgb_cv >= lgb_cv:
        best_type = "xgb"
        best_name = "XGBoost"
        best_params = xgb_params
        best_cv = xgb_cv
    else:
        best_type = "lgb"
        best_name = "LightGBM"
        best_params = lgb_params
        best_cv = lgb_cv

    print(
        f"\nSelected model: {best_name}"
    )

    print(
        f"CV PR-AUC: {best_cv:.4f}"
    )

    # -------------------------------------------------------------------------
    # MODEL BEFORE SHAP PRUNING
    # -------------------------------------------------------------------------
    pre_shap_model = fit_model(
        best_type,
        best_params,
        X_train,
        y_train
    )

    # Threshold for TRAIN evaluation is obtained from OOF predictions.
    oof_before = oof_predictions(
        best_type,
        best_params,
        X_train,
        y_train,
        splits
    )

    y_oof_before = y_train.loc[
        oof_before.index
    ]

    (
        threshold_before,
        _,
        _,
        _
    ) = best_fbeta_threshold(
        y_oof_before,
        oof_before,
        FBETA_BETA
    )

    print(
        "\nBEFORE SHAP PRUNING"
    )

    evaluate(
        pre_shap_model,
        X_train,
        y_train,
        threshold_before,
        "TRAIN - BEFORE SHAP PRUNING"
    )

    # -------------------------------------------------------------------------
    # SHAP
    # -------------------------------------------------------------------------
    top_features, shap_df = shap_top_18(
        pre_shap_model,
        X_train
    )

    print(
        "\n" + "=" * 70
    )
    print(
        "TOP 18 SHAP FEATURES"
    )
    print(
        "=" * 70
    )

    print(
        shap_df.head(18).to_string(
            index=False
        )
    )

    # Save SHAP table
    shap_df.head(18).to_csv(
        "FINAL_18_SHAP_FEATURES.csv",
        index=False
    )

    # -------------------------------------------------------------------------
    # PRUNED DATA
    # -------------------------------------------------------------------------
    X_train_18 = X_train[
        top_features
    ].copy()

    X_test_18 = X_test[
        top_features
    ].copy()

    X_score_18 = X_score[
        top_features
    ].copy()

    print(
        "\nFinal number of features:",
        len(top_features)
    )

    # -------------------------------------------------------------------------
    # FINAL TUNING ON 18 FEATURES
    # -------------------------------------------------------------------------
    xgb_params_18, xgb_cv_18 = tune_model(
        "xgb",
        X_train_18,
        y_train,
        splits,
        n_trials=20
    )

    lgb_params_18, lgb_cv_18 = tune_model(
        "lgb",
        X_train_18,
        y_train,
        splits,
        n_trials=20
    )

    if xgb_cv_18 >= lgb_cv_18:
        final_type = "xgb"
        final_name = "XGBoost"
        final_params = xgb_params_18
    else:
        final_type = "lgb"
        final_name = "LightGBM"
        final_params = lgb_params_18

    # -------------------------------------------------------------------------
    # FINAL OOF THRESHOLD
    # -------------------------------------------------------------------------
    oof_final = oof_predictions(
        final_type,
        final_params,
        X_train_18,
        y_train,
        splits
    )

    y_oof_final = y_train.loc[
        oof_final.index
    ]

    (
        final_threshold,
        threshold_precision,
        threshold_recall,
        threshold_fbeta
    ) = best_fbeta_threshold(
        y_oof_final,
        oof_final,
        FBETA_BETA
    )

    print(
        "\n" + "=" * 70
    )
    print(
        "FINAL THRESHOLD"
    )
    print(
        "=" * 70
    )
    print(
        f"Threshold: {final_threshold:.4f}"
    )
    print(
        f"OOF Precision: {threshold_precision:.4f}"
    )
    print(
        f"OOF Recall: {threshold_recall:.4f}"
    )
    print(
        f"OOF F1.5: {threshold_fbeta:.4f}"
    )

    # -------------------------------------------------------------------------
    # FINAL MODEL
    # -------------------------------------------------------------------------
    final_model = fit_model(
        final_type,
        final_params,
        X_train_18,
        y_train
    )

    # -------------------------------------------------------------------------
    # TRAIN AFTER SHAP PRUNING
    # -------------------------------------------------------------------------
    evaluate(
        final_model,
        X_train_18,
        y_train,
        final_threshold,
        "TRAIN - AFTER SHAP PRUNING - FINAL 18 FEATURES"
    )

    # -------------------------------------------------------------------------
    # FINAL TEST EVALUATION
    # -------------------------------------------------------------------------
    test_probability, test_prediction = evaluate(
        final_model,
        X_test_18,
        y_test,
        final_threshold,
        "TEST - FINAL 18-FEATURE MODEL"
    )

    # -------------------------------------------------------------------------
    # FINAL ROC / PR CURVES
    # -------------------------------------------------------------------------
    roc_auc_final, pr_auc_final = (
        plot_final_curves(
            y_test,
            test_probability
        )
    )

    print(
        "\nFinal TEST ROC-AUC:",
        round(roc_auc_final, 4)
    )

    print(
        "Final TEST PR-AUC:",
        round(pr_auc_final, 4)
    )

    # -------------------------------------------------------------------------
    # Q11 SCORING
    # -------------------------------------------------------------------------
    q11_probability = (
        final_model.predict_proba(
            X_score_18.fillna(-999)
        )[:, 1]
    )

    q11_prediction = (
        q11_probability
        >= final_threshold
    ).astype(int)

    output = pd.DataFrame({
        ID_COL: score_merged[ID_COL],
        "adoption_probability_Q11":
            q11_probability,
        "predicted_adoption_Q11":
            q11_prediction
    })

    output = output.sort_values(
        "adoption_probability_Q11",
        ascending=False
    ).reset_index(drop=True)

    output["adoption_rank"] = (
        np.arange(len(output)) + 1
    )

    output.to_csv(
        OUTPUT_PATH,
        index=False
    )

    # -------------------------------------------------------------------------
    # FINAL SUMMARY
    # -------------------------------------------------------------------------
    print(
        "\n" + "=" * 70
    )
    print(
        "FINAL SUMMARY"
    )
    print(
        "=" * 70
    )

    print(
        f"Candidate features before SHAP: "
        f"{len(feature_cols)}"
    )

    print(
        f"Final SHAP-selected features: "
        f"{len(top_features)}"
    )

    print(
        f"Final model: {final_name}"
    )

    print(
        f"Final threshold: "
        f"{final_threshold:.4f}"
    )

    print(
        f"Final TEST ROC-AUC: "
        f"{roc_auc_final:.4f}"
    )

    print(
        f"Final TEST PR-AUC: "
        f"{pr_auc_final:.4f}"
    )

    print(
        f"Q11 predicted adopters: "
        f"{q11_prediction.sum()}"
    )

    print(
        f"Q11 total physicians: "
        f"{len(q11_prediction)}"
    )

    print(
        f"\nOutput: {OUTPUT_PATH}"
    )

    print(
        "\nFinal 18 features:"
    )

    for i, feature in enumerate(
        top_features,
        1
    ):
        shap_value = shap_df.loc[
            shap_df["feature"] == feature,
            "mean_abs_shap"
        ].iloc[0]

        print(
            f"{i:2d}. {feature:50s} "
            f"{shap_value:.6f}"
        )

    return {
        "model": final_model,
        "features": top_features,
        "shap": shap_df.head(18),
        "threshold": final_threshold,
        "test_probability": test_probability,
        "test_prediction": test_prediction,
        "q11_output": output
    }


if __name__ == "__main__":
    results = main()